In [1]:
Output = c("/Users/alexis/Library/CloudStorage/OneDrive-UniversityofNorthCarolinaatChapelHill/CEMALB_DataAnalysisPM/Projects/P1014. Nanoparticles Respiratory Tract/P1014.3. Analyses/P1014.3.1. Group Distribution Analysis/Output")
cur_date = "021326"

library(readxl)
library(openxlsx)
library(tidyverse)
library(reshape2)
library(rlang)
library(limma)

# reading in files
protein_df = data.frame(read_excel("Input/Protein_Data_020926.xlsx", sheet = 2))[,4:13]
protein_df$Dose = as.numeric(protein_df$Dose)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths



Attaching package: ‘rlang’


The following objects are masked from ‘package:purrr’:

    %@%, flatten, flatten_chr, flatten_dbl, flatten_int, flatten_lgl,
    flatten_raw, invoke, splice




In [2]:
head(protein_df)

,Subject_No,Treatment,Sample_ID,Dose,Separation,ELISA_ID,UniProt_ID,Protein_Name,Conc,Conc_pslog2
,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
1,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03001,Q07011,4-1BB,2.002402,1.586117
2,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03541,Q15109,AGER,3.425807,2.145941
3,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL01701,Q9UNG2,AITRL (GITR Ligand),2.178579,1.668382
4,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL04271,P31749,AKT1,2.214532,1.684609
5,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03101,Q15389,Angiopoietin-1,2.611234,1.852492
6,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL09731,P05089,ARG1,3.369304,2.127404


In [3]:
# only interested in centrifuged samples
protein_df = protein_df %>%
    filter(Separation == 'C')

we are interested in comparing NP1_Rhodamine B and NP2 with Vehicle control. For NP1 and NP1_Rhodamine B, two doses are available. 
Let's answer this with a 

Now on the data analysis side, we are interested in dose-dependent effect of NP1 and NP1_Rhodamine B. A comparison between NP1 and NP1_Rhodamine B will also be informative. Ultimately, the effect of NP2 and NP2_Rhodamine B (with only n=1) at the higher dose will be very informative. Comparison between NP1 and NP2 at the higher dose will be informative too.


## Is there a statistically significant difference in protein expression in respiratory tract cells based on treatment? How does this expression change with dose?

We'll use linear modeling to compare, NP1 (at both doses), NP1-Rhodamine B (at both doses) and NP2 to controls.

In [7]:
# converting to a wide format to run limma
split_protein_df = protein_df %>%
    group_by(Treatment) %>%
    group_split()

control_df = split_protein_df[[1]]
np1_df = split_protein_df[[2]]
np1r_df = split_protein_df[[3]]
np2_df = split_protein_df[[4]]

wide_matrix = function(split_df, control_df){

    # recombining the control data with all other tx groups
    combined_df = rbind(split_df, control_df)
    
    wide_version = combined_df %>%
        select(Sample_ID, UniProt_ID, Conc_pslog2) %>%
        pivot_wider(names_from = Sample_ID, values_from = Conc_pslog2) %>%
        column_to_rownames("UniProt_ID") %>%
        as.matrix()

    return(wide_version)
}

# calling fn
wide_np1_matrix = wide_matrix(np1_df, control_df)
wide_np1r_matrix = wide_matrix(np1r_df, control_df)
wide_np2_matrix = wide_matrix(np2_df, control_df)

head(wide_np1_matrix)

,NP1_3_0.1,NP1_6_0.05,NP1_11_0.1,NP1_14_0.05,NP1_20_0.1,NP1_24_0.05,Control_2_NA,Control_10_NA,Control_17_NA
Q07011,1.569602,1.572613,1.605838,1.554672,1.592782,1.576927,1.585350,1.583870,1.575114
Q15109,2.138565,2.143017,2.129192,2.148347,2.115151,2.139980,2.135693,2.073869,2.138415
Q9UNG2,1.654883,1.646805,1.661696,1.650657,1.664520,1.660484,1.642756,1.674437,1.629017
P31749,1.680710,1.663189,1.660594,1.667402,1.671090,1.684437,1.670583,1.663760,1.649666
Q15389,1.773931,1.803776,1.776046,1.791555,1.783841,1.810487,1.896098,1.877910,1.851314
P05089,2.121573,2.109544,2.128535,2.149856,2.133167,2.132945,2.120742,2.113760,2.121512


In [10]:
# creating a sample info df
create_sample_info = function(wide_matrix, Variable){

    sample_info_df = protein_df[,2:4] %>%
        unique() %>%
        filter(Treatment %in% c("Control", Variable)) %>%
        # ensuring the sample ids are in the same order as the previous df
        arrange(match(Sample_ID, colnames(wide_matrix)))

    return(sample_info_df)
    
    }

# calling fn
np1_sample_df = create_sample_info(wide_np1_matrix, "NP1")
np1r_sample_df = create_sample_info(wide_np1r_matrix, "NP1-Rhodamine B")
np2_sample_df = create_sample_info(wide_np2_matrix, "NP2")

head(np1_sample_df)

,Treatment,Sample_ID,Dose
,<chr>,<chr>,<dbl>
1,NP1,NP1_3_0.1,0.10
2,NP1,NP1_6_0.05,0.05
3,NP1,NP1_11_0.1,0.10
4,NP1,NP1_14_0.05,0.05
5,NP1,NP1_20_0.1,0.10
6,NP1,NP1_24_0.05,0.05


Here we want to see how dose affects treatment's effect on protein expression, hence the interaction. ie. Treatment + Dose

If we wanted to control for dose, it would help us determine treatment's effect *regardless* of dose. ie. Treatment * Dose

Can't run an interaction for a dependent variable with only one dose, so to keep it consistent across all treatments I'll just stratify to see how dose modifies protein expression too. 

In [20]:
# running limma
# comparing each tx group to the control
design = model.matrix(~ Treatment * Dose, data = np1_sample_df)
linear_fit = lmFit(wide_np1_matrix, design)
limma_fit = eBayes(linear_fit, robust = TRUE)

treatments = colnames(design)[2:length(colnames(design))]
limma_df = data.frame()
for (i in 1:length(treatments)){

    # BH adjusted p value

    limma_results = topTable(limma_fit, coef = treatments[i], number = Inf) %>%
        rownames_to_column("UniProt_ID") %>%
        mutate(Treatment = treatments[i]) 

    limma_df = rbind(limma_df, limma_results)

    }

# cleaning up the df
limma_df = limma_df[,c(1,2,5,6,8)] %>%
    mutate(Treatment = ifelse(grepl("Treatment", Treatment), 
                    unlist(strsplit(Treatment, split = "Treatment"))[2], Treatment))
colnames(limma_df)[3:4] = c("P Value", "P Adj")

# adding in meta data
final_df = inner_join(limma_df, unique(protein_df[,6:8]))[,c(6,1,7,5,2:4)]
head(final_df)

Coefficients not estimable: TreatmentNP1:Dose 


Warning message:
“Partial NA coefficients for 280 probe(s)”
Warning message in .ebayes(fit = fit, proportion = proportion, stdev.coef.lim = stdev.coef.lim, :
“Estimation of var.prior failed - set to default value”
Joining with `by = join_by(UniProt_ID)`


,ELISA_ID,UniProt_ID,Protein_Name,Treatment,logFC,P Value,P Adj
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,nEL03731,P10909,CLU,NP1,-0.26121579,2.116014e-08,5.924840e-06
2,nEL00061,P10147,CCL3,NP1,0.18596399,3.426870e-07,4.797619e-05
3,nEL04021,O00300,TNFRSF11B,NP1,-0.16012215,1.065162e-04,9.941513e-03
4,nEL01981,Q99988,GDF-15 (MIC-1),NP1,0.08672128,1.859697e-03,1.132655e-01
5,nEL03211,P43489,TNFRSF4 (OX40),NP1,0.08215119,2.107513e-03,1.132655e-01
6,nEL12471,P05107,ITGB2,NP1,0.08151142,2.427119e-03,1.132655e-01


In [21]:
limma_df %>%
    filter(`P Adj` < 0.05)

UniProt_ID,logFC,P Value,P Adj,Treatment
<chr>,<dbl>,<dbl>,<dbl>,<chr>
P10909,-0.2612158,2.116014e-08,5.924840e-06,NP1
P10147,0.1859640,3.426870e-07,4.797619e-05,NP1
O00300,-0.1601221,1.065162e-04,9.941513e-03,NP1
O00300,-3.0518350,7.513341e-07,2.103736e-04,Dose
P10909,-1.7309694,8.111718e-06,1.135641e-03,Dose
Q9H2A7,-1.5854498,5.883462e-05,5.491231e-03,Dose
P10147,-0.9844980,6.067617e-04,4.247332e-02,Dose


In [23]:
with(sample_info_df, table(Treatment, Dose, useNA="ifany"))
limma::is.fullrank(model.matrix(~ Treatment * Dose, data=sample_info_df))

         Dose
Treatment 0 0.05 0.1
  Control 3    0   0
  NP1     0    3   3

[1] FALSE